# Data Load

In [50]:
import pandas as pd

retail_df = pd.read_csv('./data/online_retail_II.csv')
retail_df

retail_df['Sales'] = retail_df['Quantity'] * retail_df['Price']
retail_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


## Data Information

In [33]:
retail_df.info()

retail_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
 8   Sales        1067371 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 73.3+ MB


,Quantity,Price,Customer ID,Sales
count,1.067371e+06,1.067371e+06,824364.000000,1.067371e+06
mean,9.938898e+00,4.649388e+00,15324.638504,1.806987e+01
std,1.727058e+02,1.235531e+02,1697.464450,2.924202e+02
min,-8.099500e+04,-5.359436e+04,12346.000000,-1.684696e+05
25%,1.000000e+00,1.250000e+00,13975.000000,3.750000e+00
50%,3.000000e+00,2.100000e+00,15255.000000,9.900000e+00
75%,1.000000e+01,4.150000e+00,16797.000000,1.770000e+01
max,8.099500e+04,3.897000e+04,18287.000000,1.684696e+05


## Data Issues

### **Missing Value Identification**

In [46]:
missing_count = retail_df.isnull().sum()

if missing_count.any() > 0:
    print(
        f"There are null values in the dataset. \nThe columns with null values are: {missing_count[missing_count > 0].index.tolist()} with missing count: {missing_count[missing_count > 0].values.tolist()}"
    )
else:
    print("There are no null values in the dataset.")

There are null values in the dataset. 
The columns with null values are: ['Description', 'Customer ID'] with missing count: [4382, 243007]


### **The Missing Value Percentage**

In [22]:
print(f"The missing data percentage is: {missing_count.sum() / len(retail_df) * 100:.2f}%")

The missing data percentage is: 23.18%


### **Duplicate Data Problem**

In [28]:
duplicate_values = retail_df.duplicated().sum()

if duplicate_values :
    print(f"There are {duplicate_values} duplicate values present in the dataset.")
else:
    print("There are no duplicate values present in the dataset.")

There are 34335 duplicate values present in the dataset.


### **The Duplicate Data Percentage**

In [29]:
duplicate_percentage = (duplicate_values / len(retail_df)) * 100
print(f"The duplicate data percentage is: {duplicate_percentage:.2f}%")

The duplicate data percentage is: 3.22%


## ***Returning a Updated Dataset as CSV***

In [57]:
retail_df_updated = retail_df.dropna(subset=["Customer ID"]).reset_index(drop=True)

retail_df_updated.to_csv('./data/online_retail_II_updated.csv', index=False)

## Checking the Exact Duplicates from the Updated Dataset 

In [60]:
retail_df_updated.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country', 'Sales'],
      dtype='object')

In [63]:
retail_df_updated.duplicated().sum()

26479

In [67]:
check_duplicates = retail_df_updated.duplicated(
    subset=["Invoice", "StockCode", "Quantity", "Price", "InvoiceDate", "Customer ID"]
)
# check_duplicates.sum()
print("Before Deletion: ",retail_df_updated)

retail_df_v2 = retail_df_updated.drop_duplicates()
print("After Deletion: ", retail_df_v2)

Before Deletion:         Invoice StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...        ...       ...                                  ...       ...   
824359  581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
824360  581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
824361  581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
824362  581587     22138        BAKING SET 9 PIECE RETROSPOT          3   
824363  581587      POST                              POSTAGE         1   

                InvoiceDate  Price  Customer ID         Country   Sales  
0      

In [68]:
retail_df_v2.to_csv('./data/online_retail_II_v2.csv', index=False)

### **Verify Invoice Uniqueness per Customer** *(Data Corruption)*

In [70]:
invoice_cust_check = retail_df_v2.groupby("Invoice")["Customer ID"].nunique()
bad_invoices = invoice_cust_check[invoice_cust_check > 1]

if len(bad_invoices) > 0:
    print(f"⚠️ WARNING: {len(bad_invoices)} Invoices linked to multiple customers!")
    # Optional: Drop these ambiguous invoices
    # df = df[~df['Invoice'].isin(bad_invoices.index)]
else:
    print("✅ All Invoices map to exactly one Customer.")

✅ All Invoices map to exactly one Customer.
